# 03-FastAPI & REST Physics

In Modules 01 and 02, we built Frontends. Gradio and Streamlit are monolithic frameworks designed to generate HTML, CSS, and JavaScript so that **humans** can interact with your PyTorch models.

But in a true Enterprise environment, humans rarely talk directly to the GPU server.
If your company builds an iOS app, a smart refrigerator, or a massive data pipeline, those systems do not have web browsers. They are machines. To allow machines to communicate with your Neural Networks across the planet, we must build a **REST API (Representational State Transfer Application Programming Interface)**.

Today, we leave the UI behind and master **FastAPI**—the undisputed industry standard for serving Python-based Artificial Intelligence.

Let's set up our environment to build the HTTP bridge.

In [1]:
import torch
import torch.nn as nn
from fastapi import FastAPI
from pydantic import BaseModel
import time
import json
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch, FastAPI, and Pydantic Environment Ready.")


✅ PyTorch, FastAPI, and Pydantic Environment Ready.


# 1. The Physics of HTTP & REST

At its core, a Neural Network is just a mathematical function: $\hat{y} = f(x)$.

* $x$ is the input tensor.
* $f$ is the PyTorch model (the weights and biases).
* $\hat{y}$ is the output tensor.

**REST** is a set of architectural rules that allows us to execute $f(x)$ over the internet using the **HTTP protocol**.

When a machine wants your model to process data, it sends an HTTP Request. For AI engineering, we only care about two HTTP "Verbs":

1. **`GET`**: Used to retrieve information. (e.g., *"Is the GPU server currently online?"*). `GET` requests cannot contain large payloads of data.
2. **`POST`**: Used to submit data for processing. (e.g., *"Here is a 500-word paragraph, please run your LLM and return a summary."*) **99% of all Machine Learning inference endpoints are `POST` routes.**

# 2. The Mathematical Cost of Serialization (JSON Bloat)

You cannot send a PyTorch Tensor object over an Ethernet cable. Tensors only exist inside your server's RAM/VRAM. To send data over the internet, it must be serialized into a universal text format: **JSON (JavaScript Object Notation)**.

This introduces a massive physics problem for AI Engineers: **Serialization Bloat**.

### The Calculus of Data Expansion

Inside PyTorch, a single `float32` number requires exactly **4 Bytes** of RAM.


$$Memory(x_{tensor}) = 4 \text{ bytes}$$

If we extract that number (e.g., `0.8413729`) and serialize it into a JSON string to send over HTTP, it becomes text. In text, every single character is 1 byte.
The string `"0.8413729"` contains 9 characters. Add a comma and a space for the JSON array (`"0.8413729", `), and it takes **11 Bytes**.

$$Memory(x_{JSON}) \approx 11 \text{ bytes}$$

**The Impact**: When you convert a 1,000-dimensional continuous embedding vector from a PyTorch Tensor to a JSON HTTP payload, the data size physically explodes by nearly **300%**. As an AI Backend Engineer, you must deeply understand this overhead, as it directly impacts your network bandwidth and API latency.

# 3. Why FastAPI? (ASGI vs WSGI)

For a decade, Python developers used Flask or Django to build APIs. These rely on **WSGI (Web Server Gateway Interface)**, which is strictly synchronous.
If 100 users hit a Flask API, and the PyTorch model takes 1 second to process an image, User 100 has to wait in a blocked queue for 100 seconds to get their result.

FastAPI is built on **ASGI (Asynchronous Server Gateway Interface)**. It natively supports `async / await` operations. While the GPU is busy crunching a massive matrix multiplication for User 1, the FastAPI server can concurrently handle incoming requests, validate JSON payloads, and manage connections for Users 2 through 100 without blocking the main event loop.

# 4. Architecting a Machine Learning API

Let's build a fully functional, production-ready REST API. We will simulate a Text-to-Embedding model. The API will receive a JSON string, convert it to a Tensor, simulate an NLP encoding layer, and return a JSON array of floats.

*(Note: In a real environment, you run this file from the terminal using `uvicorn main:app --host 0.0.0.0 --port 8000`. For this notebook, we will architect the exact code you would put in `main.py`)*

In [2]:
# --- ⚙️ main.py ---
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import torch
import torch.nn as nn
import time

# 1. Initialize the FastAPI Application
app = FastAPI(
    title="Enterprise Embedding API",
    description="A high-performance REST API for generating semantic vector embeddings.",
    version="1.0.0"
)

# 2. Architect the Data Schemas (Pydantic)
# We strictly define the JSON structure we expect from the client.
# If a client sends a missing field or an integer instead of a string, FastAPI rejects it instantly!
class InferenceRequest(BaseModel):
    text: str
    model_version: str = "v1-base"

class InferenceResponse(BaseModel):
    embedding: list[float]
    inference_time_ms: float

# 3. Initialize the PyTorch Model (Simulated)
class MockEmbeddingModel(nn.Module):
    def __init__(self, vocab_size=1000, hidden_dim=8):
        super().__init__()
        # A tiny simulated embedding matrix
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        
    def forward(self, text_length: int):
        # Simulate processing length-dependent context
        simulated_tokens = torch.randint(0, 1000, (1, text_length))
        # Extract embeddings and average them to get a single document vector
        doc_vector = self.embedding(simulated_tokens).mean(dim=1).squeeze()
        return doc_vector

print("Loading PyTorch Weights into Server Memory...")
# In production, this happens ONCE when the server boots up.
ai_model = MockEmbeddingModel()
ai_model.eval() 

# 4. Define the Health Check Route (GET)
# Kubernetes and Load Balancers ping this constantly to ensure the server hasn't crashed.
@app.get("/health")
async def health_check():
    return {"status": "healthy", "gpu_available": torch.cuda.is_available()}

# 5. Define the Neural Inference Route (POST)
@app.post("/v1/embeddings", response_model=InferenceResponse)
async def generate_embedding(request: InferenceRequest):
    start_time = time.time()
    
    # A. Pre-Processing & Validation
    if len(request.text) < 3:
        # HTTP 400 Bad Request if the input is too short
        raise HTTPException(status_code=400, detail="Text sequence too short for semantic mapping.")
    
    text_length = len(request.text.split())
    
    # B. The Mathematical Core (GPU/CPU Inference)
    with torch.no_grad():
        # Execute the model
        tensor_output = ai_model(text_length)
        
    # C. Tensor Serialization
    # We must convert the PyTorch Tensor back into standard Python floats for JSON conversion
    json_ready_list = tensor_output.tolist()
    
    calc_time = (time.time() - start_time) * 1000.0 # Convert to milliseconds
    
    # D. Return the formatted payload
    return InferenceResponse(
        embedding=json_ready_list,
        inference_time_ms=calc_time
    )

print("Success! FastAPI Server Compiled. Routing architecture is active.")

Loading PyTorch Weights into Server Memory...
Success! FastAPI Server Compiled. Routing architecture is active.


# 5. Simulating the Machine-to-Machine Interaction

Because this API is not a web page, you cannot "click" on it. We must write a Python script acting as the **Client Machine** (e.g., an iOS App backend) sending an HTTP payload over the network.

We will simulate the exact JSON request and response physics.

In [ ]:
# --- 📡 Client Simulation Script ---
import json

print("\n--- 🌐 Initiating HTTP POST Request ---")
# 1. The Client constructs the JSON Payload
client_payload = {
    "text": "Deep Learning enables massive REST APIs.",
    "model_version": "v1-base"
}
json_payload = json.dumps(client_payload, indent=2)
print(f"Client sends JSON Bytes over network:\n{json_payload}")

# [SIMULATED HTTP NETWORK TRANSMISSION]

# 2. The Server receives the payload, executes the PyTorch Model, and returns a response
# (We bypass the actual network port here and just call the Python function directly to simulate)
import asyncio
# FastAPI uses asynchronous routes, so we must run it in an event loop
try:
    # Convert dict to Pydantic object manually for the simulation
    request_obj = InferenceRequest(**client_payload)
    response_obj = await generate_embedding(request_obj)
    
    # 3. The Server serializes the output and sends it back to the client
    server_json_response = json.dumps(response_obj.model_dump(), indent=2)
    print(f"\n--- 📥 Server Returns JSON Response ---")
    print(server_json_response)
    
except Exception as e:
    print(f"HTTP Error: {e}")

print("\n--- 💡 Architecture Insight ---")
print("Look at the Server Response. The complex, continuous PyTorch Tensor was successfully serialized into an array of floats. A separate machine written in Node.js, Ruby, or Go can now receive this array, parse it, and use it to search a Vector Database, without ever needing to know what PyTorch is!")


--- 🌐 Initiating HTTP POST Request ---
Client sends JSON Bytes over network:
{
  "text": "Deep Learning enables massive REST APIs.",
  "model_version": "v1-base"
}

--- 📥 Server Returns JSON Response ---
{
  "embedding": [
    -0.28417569398880005,
    0.5296773910522461,
    -0.3587229251861572,
    -0.23555874824523926,
    0.27817651629447937,
    0.40776923298835754,
    0.10440248250961304,
    0.46624627709388733
  ],
  "inference_time_ms": 141.65163040161133
}

--- 💡 Architecture Insight ---
Look at the Server Response. The complex, continuous PyTorch Tensor was successfully serialized into an array of floats. A separate machine written in Node.js, Ruby, or Go can now receive this array, parse it, and use it to search a Vector Database, without ever needing to know what PyTorch is!


## Real-World Use Case or Analogy:

Think of the difference between a Streamlit UI and a FastAPI REST backend like **A Sit-Down Restaurant vs. A Wholesale Drive-Thru**:

* **Streamlit (The Sit-Down Restaurant)**: Streamlit generates HTML for humans. The user walks in, sits down, looks at a menu, and talks to a waiter. It is highly visual, comfortable, and stateful. But you can only serve humans one at a time. If another *business* wants to buy 1,000 burgers from you, they aren't going to sit at 1,000 tables.
* **FastAPI (The Wholesale Drive-Thru)**: FastAPI generates JSON for machines. It has no tables, no decorations, and no waiters. It is a high-speed, concrete drive-thru window. A massive truck (another server) pulls up, violently hands the cashier a standardized clipboard of text (The JSON POST Request), the kitchen throws a box of 1,000 burgers through the window (The JSON Response), and the truck drives away in 50 milliseconds. It is purely designed for ruthless, high-volume machine-to-machine integration.